In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from databricks.feature_engineering  import FeatureEngineeringClient

In [0]:
data = {
'sexo': ['Hombre', 'Mujer', 'No_Declarado', 'Hombre', 'Hombre', 'Mujer', 
             'No_Declarado', 'Mujer', 'Hombre', 'Mujer', 'Hombre'],
'nivel_educativo': ['Bachillerato', 'Licenciatura', 'Maestría', 'Licenciatura', 'Bachillerato',
'Maestría', 'Licenciatura', 'Bachillerato', 'Bachillerato', 'Licenciatura', 'Licenciatura'],
'edad': [25, 45, 65, 35, 22,67,20,31,45,36,70],
        'salario': [20000, 50000, 100000, 35000, 27000,40000,15000,18000,13000,40000,21000]
}

data=list(zip(data["sexo"],data["nivel_educativo"],data["edad"],data["salario"]))
df_train = spark.createDataFrame(data,["sexo","nivel_educativo","edad","salario"])
df_train.display()

In [0]:
data_test = {
'sexo': ['Hombre', 'Mujer', 'No_Declarado','Mujer',"No sé"],
'nivel_educativo': ['Secundaria', 'Licenciatura', 'Maestría','Bachillerato'],
'edad': [18,29,75,39],
        'salario': [8000,15000,17000,25000]
}

data_test=list(zip(data_test["sexo"],data_test["nivel_educativo"],data_test["edad"],data_test["salario"]))
df_test = spark.createDataFrame(data_test,["sexo","nivel_educativo","edad","salario"])
df_test.display()

In [0]:
from pyspark.ml.feature import OneHotEncoder,StringIndexer,VectorAssembler,CountVectorizer
from pyspark.ml import Pipeline

In [0]:
string_idx=StringIndexer(inputCols=["sexo","nivel_educativo"],outputCols=["idx_sexo","idx_nivel_educativo"],handleInvalid="keep",stringOrderType="alphabetAsc")
ohe=OneHotEncoder(inputCol="idx_sexo",outputCol="ohe_sexo",dropLast=False)
pipe=Pipeline(stages=[string_idx,ohe])
pipe.fit(df_train).transform(df_train).display()
[1,0,0,0]

In [0]:
string_idx=StringIndexer(inputCols=["sexo","nivel_educativo"],outputCols=["idx_sexo","idx_nivel_educativo"],handleInvalid="skip",stringOrderType="alphabetAsc")
"""
skip,error
"frequencyDesc"categorías mas frecuentes tienen índices más bajos. 
"frequencyAsc"  categorías menos frecuentes tienen índices más bajos.
"alphabetDesc"  orden alfabético inverso.
"alphabetAsc"  orden alfabético .
"""
ohe=OneHotEncoder(inputCols=["idx_sexo"],outputCols=["ohe_sexo"],dropLast=False)
assembler = VectorAssembler(
    inputCols=["idx_nivel_educativo","ohe_sexo"], 
    outputCol="features"
)
pip=Pipeline(stages=[string_idx,ohe,assembler])
df_ohe=pip.fit(df_train).transform(df_test)
df_ohe.display()

In [0]:
from pyspark.ml.feature import StandardScaler,MinMaxScaler

In [0]:
vec_assembler = VectorAssembler(inputCols=["salario"], outputCol="salario_vec")
standar_sca=StandardScaler(inputCol="salario_vec",outputCol="std_salario",withMean=True, withStd=True)
norm_salario=MinMaxScaler(inputCol="salario_vec",outputCol="norm_salario")
pipe=Pipeline(stages=[vec_assembler,standar_sca,norm_salario])
pipe_salario=pipe.fit(df_train).transform(df_test)
pipe_salario.display()

In [0]:
from pyspark.ml.feature import StandardScaler,MinMaxScaler, Bucketizer, QuantileDiscretizer,CountVectorizer

El algoritmo debe encontrar los puntos de corte (p) para el 33.33% (1/3) y el 66.67% (2/3) de la distribución. La fórmula de posición empírica básica que rige esto es:
$i=p⋅N$, entonces para $$i_1=\dfrac{1}{3}\times 11$$

La fórmula matemática para encontrar el cuantil exacto es :
$$
Q=X_k​+d⋅(X_{k+1}​−X_k​)
$$
Donde $X_k$​ es el valor en la posición 3, y Xk+1​ es el valor en la posición 4 y d la parte decimal

In [0]:
qd = QuantileDiscretizer(
    numBuckets=3,
    inputCol="edad",
    outputCol="edad_qtile",
 ).fit(df_train)

df_q = qd.transform(df_test)
df_q.select("edad", "edad_qtile").display()

In [0]:
from pyspark.sql import functions as F
splits = [-float("inf"), 30.0, 60.0, float("inf")]

bucketizer = Bucketizer(
    inputCol="edad",
    outputCol="edad_bin",
    splits=splits
)

df_binned = bucketizer.transform(df_train)

labels = F.when(F.col("edad_bin") == 0.0, F.lit("joven")) \
          .when(F.col("edad_bin") == 1.0, F.lit("adulto")) \
          .otherwise(F.lit("mayor"))

df_binned = df_binned.withColumn("edad_rango", labels)

df_binned.select("edad", "edad_bin", "edad_rango").display()

In [0]:
data = [
    (1, ["http", "tcp"]),
    (2, ["tls", "tcp"]),
    (3, ["dns", "udp"])
]

df = spark.createDataFrame(data, ["id", "protocols"])

vectorizer = CountVectorizer(inputCol="protocols", outputCol="protocols_vec", binary=True)
model = vectorizer.fit(df)
df_vectorized = model.transform(df)

df_vectorized.select("id", "protocols_vec").display(truncate=False)

In [0]:
vectorizer.

In [0]:
with mlflow.start_run(run_name="feature_engineering"):
    qd = QuantileDiscretizer(
    numBuckets=3,
    inputCol="edad",
    outputCol="edad_qtile",
    ).fit(df_train)

    df_q = qd.transform(df_test)

In [0]:
fe=FeatureEngineeringClient()
